In [33]:
from google.colab import files
uploaded = files.upload()

Saving recipes_raw.zip to recipes_raw.zip


In [34]:
import os
import zipfile

zip_path = "/content/recipes_raw.zip"
data_path = "/content/data"

os.makedirs(data_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(data_path)

print("Dataset extracted successfully!")
print("Files:", os.listdir(data_path))

Dataset extracted successfully!
Files: ['LICENSE', 'recipes_raw_nosource_ar.json', 'recipes_raw_nosource_epi.json', 'recipes_raw_nosource_fn.json']


In [ ]:
!pip install pandas numpy sentence-transformers faiss-cpu google-generativeai streamlit


INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of proto-plus to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 59.8 MB/s  0:

In [35]:
import json
import os

data_path = "/content/data"

json_files = [
    "recipes_raw_nosource_ar.json",
    "recipes_raw_nosource_epi.json",
    "recipes_raw_nosource_fn.json"
]

recipes = {}

for file_name in json_files:
    file_path = os.path.join(data_path, file_name)

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    for recipe_id, recipe in data.items():
        stable_id = f"{file_name.replace('.json', '')}_{recipe_id}"

        recipes[stable_id] = {
            "id": stable_id,
            "original_id": recipe_id,
            "source": file_name,
            "title": recipe.get("title"),
            "ingredients": recipe.get("ingredients"),
            "instructions": recipe.get("instructions"),
            "picture_link": recipe.get("picture_link")
        }

print("Total recipes loaded:", len(recipes))


Total recipes loaded: 125164


In [ ]:
print("Total recipes:", len(recipes))

sample_id = next(iter(recipes))
sample_recipe = recipes[sample_id]

print("\nRecipe ID:", sample_recipe["id"])
print("Title:", sample_recipe["title"])
print("Ingredients:", sample_recipe["ingredients"])
print("Instructions:", sample_recipe["instructions"])


Total recipes: 125164

Recipe ID: recipes_raw_nosource_ar_rmK12Uau.ntP510KeImX506H6Mr6jTu
Title: Slow Cooker Chicken and Dumplings
Ingredients: ['4 skinless, boneless chicken breast halves ADVERTISEMENT', '2 tablespoons butter ADVERTISEMENT', '2 (10.75 ounce) cans condensed cream of chicken soup ADVERTISEMENT', '1 onion, finely diced ADVERTISEMENT', '2 (10 ounce) packages refrigerated biscuit dough, torn into pieces ADVERTISEMENT', 'ADVERTISEMENT']
Instructions: Place the chicken, butter, soup, and onion in a slow cooker, and fill with enough water to cover.
Cover, and cook for 5 to 6 hours on High. About 30 minutes before serving, place the torn biscuit dough in the slow cooker. Cook until the dough is no longer raw in the center.



In [37]:
import sys

folder = "/content/drive/MyDrive/Recipe_Chatbot"

if folder not in sys.path:
    sys.path.append(folder)

import recipemate_clean

normalize_ingredient = recipemate_clean.normalize_ingredient

for recipe in recipes.values():
    recipe["normalized_ingredients"] = [
        normalize_ingredient(item)
        for item in (recipe.get("ingredients") or [])
        if isinstance(item, str)
    ]

print("Ingredient normalization completed!")
print("Sample:", recipes[next(iter(recipes))]["normalized_ingredients"][:5])

Ingredient normalization completed!
Sample: ['boneless chicken breast halves', 'butter', 'condensed cream chicken soup', 'onion', 'refrigerated biscuit dough']


In [39]:
import os
import pickle

folder = "/content/drive/MyDrive/Recipe_Chatbot"
os.makedirs(folder, exist_ok=True)

save_path = os.path.join(folder, "processed_recipes.pkl")

with open(save_path, "wb") as file:
    pickle.dump(recipes, file)

print("Processed recipes saved successfully!")
print("Total recipes:", len(recipes))
print("File size:", round(os.path.getsize(save_path) / (1024 * 1024), 2), "MB")

Processed recipes saved successfully!
Total recipes: 125164
File size: 213.1 MB


In [40]:
recipe_texts = []
recipe_ids = []

for recipe_id, recipe in recipes.items():
    title = recipe.get("title") or ""
    ingredients = recipe.get("normalized_ingredients") or []

    if title.strip() and ingredients:
        text = title + " " + " ".join(ingredients)
        recipe_texts.append(text)
        recipe_ids.append(recipe_id)

print("Recipes prepared:", len(recipe_texts))
print("Recipe IDs:", len(recipe_ids))
print("Sample:", recipe_texts[0][:300])

Recipes prepared: 122971
Recipe IDs: 122971
Sample: Slow Cooker Chicken and Dumplings boneless chicken breast halves butter condensed cream chicken soup onion refrigerated biscuit dough 


In [41]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    recipe_texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings generated:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/961 [00:00<?, ?it/s]

Embeddings generated: (122971, 384)


In [42]:
import faiss
import numpy as np

faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS index created successfully!")
print("Index size:", index.ntotal)
print("Dimension:", dimension)

FAISS index created successfully!
Index size: 122971
Dimension: 384


In [43]:
import os
import pickle
import faiss

folder = "/content/drive/MyDrive/Recipe_Chatbot"
os.makedirs(folder, exist_ok=True)

with open(os.path.join(folder, "recipe_embeddings.pkl"), "wb") as f:
    pickle.dump(embeddings, f)

faiss.write_index(
    index,
    os.path.join(folder, "recipe_faiss.index")
)

with open(os.path.join(folder, "recipe_ids.pkl"), "wb") as f:
    pickle.dump(recipe_ids, f)

print("All search files saved successfully!")
print("Embeddings:", embeddings.shape)
print("FAISS index:", index.ntotal)
print("Recipe IDs:", len(recipe_ids))

All search files saved successfully!
Embeddings: (122971, 384)
FAISS index: 122971
Recipe IDs: 122971


In [44]:
import re

substitution_index = {}

for recipe in recipes.values():
    instructions = recipe.get("instructions") or ""

    if isinstance(instructions, list):
        instructions = " ".join(instructions)

    text = str(instructions)

    if re.search(r"\bsubstitut\w*\b", text, re.IGNORECASE):
        substitution_index[recipe["id"]] = {
            "title": recipe.get("title"),
            "instructions": text
        }

print("Substitution index created!")
print("Recipes with substitution references:", len(substitution_index))

Substitution index created!
Recipes with substitution references: 594


In [45]:
import os
import pickle

folder = "/content/drive/MyDrive/Recipe_Chatbot"

with open(os.path.join(folder, "substitution_index.pkl"), "wb") as f:
    pickle.dump(substitution_index, f)

print("Substitution index saved successfully!")
print("Indexed recipes:", len(substitution_index))

Substitution index saved successfully!
Indexed recipes: 594


In [46]:
recipemate_clean.recipes = recipes
recipemate_clean.model = model
recipemate_clean.index = index
recipemate_clean.recipe_ids = recipe_ids
recipemate_clean.substitution_index = substitution_index

print("RecipeMate data connected successfully!")
print("Recipes:", len(recipemate_clean.recipes))
print("FAISS index:", recipemate_clean.index.ntotal)
print("Substitution records:", len(recipemate_clean.substitution_index))

RecipeMate data connected successfully!
Recipes: 125164
FAISS index: 122971
Substitution records: 594


In [47]:
response = recipemate_clean.chatbot_response(
    "Suggest a recipe using tomatoes and onions"
)

print(response)

Here are some recipes I found:

1. Tomato Sauce 
   Recipe ID: recipes_raw_nosource_epi_O7wUjhHl44xc5uxun9ymJ1vjdYXtiGy

2. Italian Zucchini Saute
   Recipe ID: recipes_raw_nosource_ar_eeSoYRx3b3lry9M5KLEeFXKHpk/8/Ke

3. Oven-Dried Tomatoes 
   Recipe ID: recipes_raw_nosource_epi_oV5jzMZyBACHJkLvwQHF//CUZv267J2

4. Salsa Fresca
   Recipe ID: recipes_raw_nosource_fn_ce.85pnc4gRQu9M01rpaJeNONhg4eS.

5. Tomato, Caramelized Onion, and Stilton Pizza
   Recipe ID: recipes_raw_nosource_fn_1C.mY1/ReikGipdFkOa1Pc4Khd.XA7.




In [54]:
recipe_id = "recipes_raw_nosource_epi_O7wUjhHl44xc5uxun9ymJ1vjdYXtiGy"

recipe = recipemate_clean.recipes[recipe_id]

print("Recipe:", recipe["title"])
print("\nIngredients:")

for ingredient in recipe["ingredients"] or []:
    print("-", ingredient)

Recipe: Tomato Sauce 

Ingredients:
- 2 medium onions, finely chopped
- 4 garlic cloves, finely chopped
- 6 tablespoons extra-virgin olive oil
- 2 (28- to 32-ounce) cans whole tomatoes in juice
- 1 teaspoon salt, or to taste
- 1 teaspoon sugar (optional), or to taste


In [55]:
response = recipemate_clean.chatbot_response(
    "Show me the ingredients of the first recipe"
)

print(response)

Recipe: Tomato Sauce 

Ingredients:
- 2 medium onions, finely chopped
- 4 garlic cloves, finely chopped
- 6 tablespoons extra-virgin olive oil
- 2 (28- to 32-ounce) cans whole tomatoes in juice
- 1 teaspoon salt, or to taste
- 1 teaspoon sugar (optional), or to taste

Instructions:
Cook onions and garlic in oil in a 4- to 5-quart heavy saucepan over moderately low heat, stirring, until softened, 6 to 8 minutes. Add tomatoes, including juice, and salt, then simmer gently, uncovered, stirring and mashing tomatoes with a fork occasionally, until sauce is thickened and reduced to about 5 cups, 1 to 1 1/4 hours. If sauce tastes too acidic, add sugar and cook 5 minutes more.
Cook onions and garlic in oil in a 4- to 5-quart heavy saucepan over moderately low heat, stirring, until softened, 6 to 8 minutes.
Add tomatoes, including juice, and salt, then simmer gently, uncovered, stirring and mashing tomatoes with a fork occasionally, until sauce is thickened and reduced to about 5 cups, 1 to 1 1

In [56]:
response = recipemate_clean.chatbot_response(
    "How long should I cook it?"
)

print(response)

NameError: name 'chat_history' is not defined

In [57]:
print("chat_history:", hasattr(recipemate_clean, "chat_history"))
print("recipes:", hasattr(recipemate_clean, "recipes"))
print("re:", hasattr(recipemate_clean, "re"))
print("Selected recipe:", recipemate_clean.chat_history.get("selected_recipe_id"))

chat_history: True
recipes: True
re: True
Selected recipe: recipes_raw_nosource_epi_O7wUjhHl44xc5uxun9ymJ1vjdYXtiGy


In [ ]:
!pip install -q nltk

In [ ]:
import re

units = [
    "tablespoons", "tablespoon", "tbsp",
    "teaspoons", "teaspoon", "tsp",
    "cups", "cup", "ounces", "ounce", "oz",
    "pounds", "pound", "lbs", "lb",
    "grams", "gram", "g", "kilograms", "kilogram", "kg",
    "milliliters", "milliliter", "ml",
    "liters", "liter", "l", "cloves", "clove"
]

containers = [
    "cans", "can", "packages", "package",
    "jars", "jar", "bottles", "bottle",
    "boxes", "box"
]

In [ ]:
def normalize_ingredient(ingredient):

    if not isinstance(ingredient, str):
        return ""

    text = ingredient.lower()

    text = re.sub(r'advertisement', '', text)
    text = re.sub(r'\([^)]*\)', '', text)

    text = re.sub(r'^\s*\d+\s+\d+/\d+\s*', '', text)
    text = re.sub(r'^\s*\d+/\d+\s*', '', text)
    text = re.sub(r'^\s*\d+(?:\.\d+)?\s*', '', text)

    text = re.sub(r'\b(skinless)\b', '', text)

    preparation_words = [
        "finely diced", "roughly chopped",
        "chopped", "minced", "diced",
        "sliced", "grated", "shredded",
        "torn into pieces"
    ]

    for word in preparation_words:
        text = text.replace(word, "")

    for unit in sorted(units, key=len, reverse=True):
        text = re.sub(rf'\b{unit}\b', '', text)

    for container in containers:
        text = re.sub(rf'\b{container}\b', '', text)

    text = re.sub(r'\bof\b', '', text)
    text = re.sub(r'[^a-z\s-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
for recipe in recipes.values():
    original_ingredients = recipe["ingredients"]

    if isinstance(original_ingredients, list):
        recipe["normalized_ingredients"] = [
            normalize_ingredient(item)
            for item in original_ingredients
            if isinstance(item, str)
        ]
    else:
        recipe["normalized_ingredients"] = []

print("Normalization completed!")
print("Total recipes:", len(recipes))

Normalization completed!
Total recipes: 125164


In [ ]:
test_ingredients = [
    "4 skinless, boneless chicken breast halves ADVERTISEMENT",
    "2 tablespoons butter ADVERTISEMENT",
    "2 (10.75 ounce) cans condensed cream of chicken soup ADVERTISEMENT",
    "1 onion, finely diced ADVERTISEMENT",
    "2 (10 ounce) packages refrigerated biscuit dough, torn into pieces ADVERTISEMENT"
]

for ingredient in test_ingredients:
    print("Original:", ingredient)
    print("Normalized:", normalize_ingredient(ingredient))
    print()

Original: 4 skinless, boneless chicken breast halves ADVERTISEMENT
Normalized: boneless chicken breast halves

Original: 2 tablespoons butter ADVERTISEMENT
Normalized: butter

Original: 2 (10.75 ounce) cans condensed cream of chicken soup ADVERTISEMENT
Normalized: condensed cream chicken soup

Original: 1 onion, finely diced ADVERTISEMENT
Normalized: onion

Original: 2 (10 ounce) packages refrigerated biscuit dough, torn into pieces ADVERTISEMENT
Normalized: refrigerated biscuit dough



In [36]:
for recipe in recipes.values():

    original_ingredients = recipe["ingredients"]

    if isinstance(original_ingredients, list):
        recipe["normalized_ingredients"] = [
            normalize_ingredient(item)
            for item in original_ingredients
            if isinstance(item, str)
        ]
    else:
        recipe["normalized_ingredients"] = []

print("Normalization completed!")
print("Total recipes:", len(recipes))

SyntaxError: invalid syntax (2984801857.py, line 9)

In [ ]:
sample_recipe = recipes[sample_id]

print("Recipe:", sample_recipe["title"])

print("\nOriginal Ingredients:")
for item in sample_recipe["ingredients"]:
    print("-", item)

print("\nNormalized Ingredients:")
for item in sample_recipe["normalized_ingredients"]:
    print("-", item)

Recipe: Slow Cooker Chicken and Dumplings

Original Ingredients:
- 4 skinless, boneless chicken breast halves ADVERTISEMENT
- 2 tablespoons butter ADVERTISEMENT
- 2 (10.75 ounce) cans condensed cream of chicken soup ADVERTISEMENT
- 1 onion, finely diced ADVERTISEMENT
- 2 (10 ounce) packages refrigerated biscuit dough, torn into pieces ADVERTISEMENT
- ADVERTISEMENT

Normalized Ingredients:
- boneless chicken breast halves
- butter
- condensed cream chicken soup
- onion
- refrigerated biscuit dough
- 


In [ ]:
import pickle

with open("/content/processed_recipes.pkl", "wb") as file:
    pickle.dump(recipes, file)

print("Processed dataset saved")

Processed dataset saved


In [ ]:
def search_recipes(query, limit=5):
    query = query.lower().strip()

    stop_words = {
        "a", "an", "the", "is", "are", "for", "with",
        "and", "or", "to", "of", "in", "on", "recipe",
        "recipes", "make", "find", "show", "me"
    }

    query_words = {
        word for word in re.findall(r'\b[a-z]+\b', query)
        if word not in stop_words
    }

    if not query_words:
        return []

    results = []

    for recipe in recipes.values():
        title = recipe.get("title") or ""
        ingredients = recipe.get("normalized_ingredients") or []

        title_words = set(re.findall(r'\b[a-z]+\b', title.lower()))

        matched_title_words = query_words.intersection(title_words)

        matched_ingredients = [
            ingredient
            for ingredient in ingredients
            if ingredient and any(
                word in ingredient.split()
                for word in query_words
            )
        ]

        matched_ingredient_words = set()

        for ingredient in ingredients:
            ingredient_words = set(ingredient.split())
            matched_ingredient_words.update(
                query_words.intersection(ingredient_words)
            )

        score = (
            5 * len(matched_title_words)
            + len(matched_ingredient_words)
        )

        if query in title.lower():
            score += 10

        if query_words.issubset(title_words):
            score += 5

        if score > 0:
            results.append({
                "id": recipe["id"],
                "title": title,
                "score": score,
                "matched_title_words": sorted(matched_title_words),
                "matched_ingredient_words": sorted(matched_ingredient_words),
                "matched_ingredients": matched_ingredients,
                "ingredients": recipe.get("ingredients") or [],
                "instructions": recipe.get("instructions") or ""
            })

    results.sort(key=lambda x: x["score"], reverse=True)

    return results[:limit]

In [ ]:
results = search_recipes("garlic", limit=5)

for recipe in results:
    print("\nRecipe:", recipe["title"])
    print("ID:", recipe["id"])
    print("Score:", recipe["score"])
    print("Matched ingredients:", recipe["matched_ingredients"])
    print("Ingredients:", recipe["ingredients"][:5])


Recipe: Garlic Chicken
ID: recipes_raw_nosource_ar_k7qcbQSKW.Cej1OLEtw3Y9GpL/Zx0dG
Score: 21
Matched ingredients: ['garlic crushed']
Ingredients: ['1/4 cup olive oil ADVERTISEMENT', '2 cloves garlic, crushed ADVERTISEMENT', '1/4 cup Italian-seasoned bread crumbs ADVERTISEMENT', '1/4 cup grated Parmesan cheese ADVERTISEMENT', '4 skinless, boneless chicken breast halves ADVERTISEMENT']

Recipe: Chicken Breasts with Balsamic Vinegar and Garlic
ID: recipes_raw_nosource_ar_8LwDZxcNVpXQAJMufYu7zXRuyfSl7pq
Score: 21
Matched ingredients: ['garlic']
Ingredients: ['4 skinless, boneless chicken breasts ADVERTISEMENT', 'salt and pepper to taste ADVERTISEMENT', '3/4 pound fresh mushrooms, sliced ADVERTISEMENT', '2 tablespoons all-purpose flour ADVERTISEMENT', '2 tablespoons olive oil ADVERTISEMENT']

Recipe: Spicy Garlic Lime Chicken
ID: recipes_raw_nosource_ar_KN9yelNk2dBtaO6hB/582QEg1hllglC
Score: 21
Matched ingredients: ['garlic powder', 'garlic powder']
Ingredients: ['3/4 teaspoon salt ADVERTI

In [ ]:
test_queries = [
    "chicken",
    "garlic pasta",
    "vegetarian dinner",
    "chocolate cake",
    "something with tomatoes"
]

for query in test_queries:
    results = search_recipes(query, limit=3)

    print("\nQuery:", query)
    for recipe in results:
        print("-", recipe["title"], "| Score:", recipe["score"])



Query: chicken
- Slow Cooker Chicken and Dumplings | Score: 21
- Chicken Pot Pie IX | Score: 21
- Ang's Balsamic Maple Chicken | Score: 21

Query: garlic pasta
- Tomato and Garlic Pasta | Score: 27
- Spinach Garlic Pasta | Score: 27
- Garlic Pasta with Prawns | Score: 27

Query: vegetarian dinner
- Vegetarian Southwest One-Pot Dinner | Score: 15
- The Best Vegetarian Chili in the World | Score: 6
- Grandma's Slow Cooker Vegetarian Chili | Score: 6

Query: chocolate cake
- Too Much Chocolate Cake | Score: 27
- German Chocolate Cake III | Score: 27
- German Chocolate Cake | Score: 27

Query: something with tomatoes
- Bow Ties with Sausage, Tomatoes and Cream | Score: 6
- Best Fried Green Tomatoes | Score: 6
- Greek Pasta with Tomatoes and White Beans | Score: 6


In [ ]:
!pip install -q sentence-transformers faiss-cpu

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [ ]:
recipe_ids = []
recipe_texts = []

for recipe in recipes.values():
    title = recipe.get("title") or ""
    ingredients = recipe.get("normalized_ingredients") or []

    if not title and not ingredients:
        continue

    text = title + " " + " ".join(ingredients)

    recipe_ids.append(recipe["id"])
    recipe_texts.append(text)

print("Recipes prepared:", len(recipe_texts))

Recipes prepared: 124595


In [ ]:
embeddings = model.encode(
    recipe_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

embeddings = embeddings.astype("float32")

print("Embedding shape:", embeddings.shape)
print("Total embeddings:", len(embeddings))

Batches:   0%|          | 0/1947 [00:00<?, ?it/s]

Embedding shape: (124595, 384)
Total embeddings: 124595


In [ ]:
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS index created successfully!")
print("Total indexed recipes:", index.ntotal)

FAISS index created successfully!
Total indexed recipes: 124595


In [ ]:
def semantic_search(query, limit=5):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, limit)

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1:
            continue

        recipe_id = recipe_ids[idx]
        recipe = recipes[recipe_id]

        results.append({
            "id": recipe_id,
            "title": recipe.get("title"),
            "similarity": round(float(score), 3),
            "ingredients": recipe.get("ingredients") or [],
            "instructions": recipe.get("instructions") or ""
        })

    return results

In [ ]:
results = semantic_search("something spicy with chicken")

for recipe in results:
    print("Title:", recipe["title"])
    print("Similarity:", recipe["similarity"])
    print("-" * 50)

Title: Spicy Chicken in Tomato-Coconut Sauce
Similarity: 0.726
--------------------------------------------------
Title: Chicken with Spicy Tomato Sauce 
Similarity: 0.724
--------------------------------------------------
Title: Spicy Duck Sauce
Similarity: 0.724
--------------------------------------------------
Title: Spiced Chicken with Oranges 
Similarity: 0.724
--------------------------------------------------
Title: Sharyl's Spicy Chicken
Similarity: 0.719
--------------------------------------------------


In [ ]:
test_queries = [
    "healthy vegetarian dinner",
    "easy chocolate cake",
    "pasta with garlic and tomatoes",
    "quick breakfast with eggs",
    "something with potatoes"
]

for query in test_queries:
    print("\nQuery:", query)

    results = semantic_search(query, limit=3)

    for recipe in results:
        print(recipe["title"], "|", recipe["similarity"])


Query: healthy vegetarian dinner
Healthy Vegetarian Recipes | 0.835
Vegetarian Southwest One-Pot Dinner | 0.658
Vegetarian Beef Shepherd's Pie | 0.647

Query: easy chocolate cake
Instant Chocolate Cake | 0.829
World's Easiest Chocolate Cake (Mud Cake)  | 0.764
The Easiest Chocolate Pudding Cake | 0.723

Query: pasta with garlic and tomatoes
Pasta With Tomatoes and Mozzarella  | 0.772
Simple Delicious Pasta Sauce | 0.76
Pasta with Grilled Tomato and Scallion Sauce  | 0.752

Query: quick breakfast with eggs
Easy Breakfast-for-Dinner Recipes | 0.737
Perfect Scrambled Eggs | 0.703
Perfect Eggs Over Easy | 0.663

Query: something with potatoes
Lemon Potatoes | 0.709
Potatoes With Chili Butter | 0.707
Potato Bites | 0.668


In [ ]:
import pickle
import os

with open("/content/recipe_embeddings.pkl", "wb") as file:
    pickle.dump(embeddings, file)

faiss.write_index(index, "/content/recipe_faiss.index")

with open("/content/recipe_ids.pkl", "wb") as file:
    pickle.dump(recipe_ids, file)

print("Embeddings and FAISS index saved successfully!")

Embeddings and FAISS index saved successfully!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil

drive_folder = "/content/drive/MyDrive/Recipe_Chatbot"

os.makedirs(drive_folder, exist_ok=True)

shutil.copy("/content/recipe_embeddings.pkl", drive_folder)
shutil.copy("/content/recipe_faiss.index", drive_folder)
shutil.copy("/content/recipe_ids.pkl", drive_folder)

print("All files saved to Google Drive")

All files saved to Google Drive


In [ ]:
def hybrid_search(query, limit=5):

    semantic_results = semantic_search(query, limit=20)
    keyword_results = search_recipes(query, limit=20)

    combined = {}

    for recipe in semantic_results:
        combined[recipe["id"]] = {
            "id": recipe["id"],
            "title": recipe["title"],
            "semantic_score": recipe["similarity"],
            "keyword_score": 0
        }

    for recipe in keyword_results:
        recipe_id = recipe["id"]

        if recipe_id not in combined:
            combined[recipe_id] = {
                "id": recipe_id,
                "title": recipe["title"],
                "semantic_score": 0,
                "keyword_score": 0
            }

        combined[recipe_id]["keyword_score"] = recipe["score"]

    results = list(combined.values())

    for recipe in results:
        recipe["final_score"] = (
            0.7 * recipe["semantic_score"]
            + 0.3 * (
                recipe["keyword_score"] /
                max([r["keyword_score"] for r in results] + [1])
            )
        )

    results.sort(key=lambda x: x["final_score"], reverse=True)

    return results[:limit]

In [ ]:
results = hybrid_search("spicy chicken with tomatoes")

for recipe in results:
    print(recipe["title"], "|", round(recipe["final_score"], 3))

Easy Spicy Chicken and Rice | 0.686
Spicy Chicken Soup | 0.668
Spicy Chicken in Tomato-Coconut Sauce | 0.543
Noodles with Spicy Tomato Sauce | 0.514
Spicy Tomato Chicken Casserole | 0.503


In [ ]:
def match_recipes_by_ingredients(user_ingredients, limit=5):

    user_ingredients = list(dict.fromkeys(
        normalize_ingredient(item)
        for item in user_ingredients
        if normalize_ingredient(item)
    ))

    results = []

    for recipe in recipes.values():

        recipe_ingredients = [
            item for item in recipe.get("normalized_ingredients", [])
            if item
        ]

        matched = []
        missing = []

        for ingredient in user_ingredients:
            found = any(
                ingredient_matches(ingredient, item)
                for item in recipe_ingredients
            )

            if found:
                matched.append(ingredient)
            else:
                missing.append(ingredient)

        if not matched:
            continue

        additional = []

        for item in recipe_ingredients:
            found = any(
                ingredient_matches(user_item, item)
                for user_item in user_ingredients
            )

            if not found:
                additional.append(item)

        match_score = len(matched) / len(user_ingredients)

        coverage = len(matched) / max(len(recipe_ingredients), 1)

        final_score = 0.7 * match_score + 0.3 * coverage

        results.append({
            "id": recipe["id"],
            "title": recipe.get("title"),
            "matched_ingredients": matched,
            "missing_ingredients": missing,
            "additional_ingredients": list(dict.fromkeys(additional)),
            "match_score": round(match_score, 2),
            "coverage": round(coverage, 2),
            "final_score": round(final_score, 3),
            "ingredients": recipe.get("ingredients") or []
        })

    results.sort(
    key=lambda x: (
        x["match_score"],
        -len(x["additional_ingredients"]),
        x["final_score"]
    ),
    reverse=True
)

    return results[:limit]

In [ ]:
results = match_recipes_by_ingredients(
    ["chicken", "tomato", "onion", "garlic"],
    limit=3
)

for recipe in results:
    print("\nRecipe:", recipe["title"])
    print("Matched:", recipe["matched_ingredients"])
    print("Missing:", recipe["missing_ingredients"])
    print("Additional:", recipe["additional_ingredients"])
    print("Final Score:", recipe["final_score"])


Recipe: Easy Chicken Cacciatore
Matched: ['chicken', 'tomato', 'onion', 'garlic']
Missing: []
Additional: ['green bell pepper', 'green beans', 'dried oregano']
Final Score: 0.871

Recipe: Quinoa Chicken
Matched: ['chicken', 'tomato', 'onion', 'garlic']
Missing: []
Additional: ['chicken broth', 'quinoa', 'vegetable oil or as needed']
Final Score: 0.871

Recipe: Spicy Chicken Soup
Matched: ['chicken', 'tomato', 'onion', 'garlic']
Missing: []
Additional: ['olive oil', 'green bell pepper', 'chicken broth']
Final Score: 0.871


In [ ]:
import re

def extract_quantity(ingredient):
    if not isinstance(ingredient, str):
        return None, None

    match = re.match(
        r'^\s*(\d+\s+\d+/\d+|\d+/\d+|\d+(?:\.\d+)?)\s*([a-zA-Z]+)?',
        ingredient
    )

    if match:
        quantity = match.group(1)
        unit = match.group(2)

        if " " in quantity:
            whole, fraction = quantity.split()
            numerator, denominator = fraction.split("/")
            quantity = float(whole) + float(numerator) / float(denominator)

        elif "/" in quantity:
            numerator, denominator = quantity.split("/")
            quantity = float(numerator) / float(denominator)

        else:
            quantity = float(quantity)

        return quantity, unit

    return None, None

In [ ]:
unit_conversions = {
    "tablespoon": "tbsp",
    "tablespoons": "tbsp",
    "tbsp": "tbsp",
    "teaspoon": "tsp",
    "teaspoons": "tsp",
    "tsp": "tsp",
    "cups": "cup",
    "cup": "cup",
    "pounds": "lb",
    "pound": "lb",
    "lbs": "lb",
    "lb": "lb",
    "ounces": "oz",
    "ounce": "oz",
    "oz": "oz",
    "cloves": "clove",
    "clove": "clove"
}

def standardize_unit(unit):
    if not unit:
        return None

    return unit_conversions.get(unit.lower(), unit.lower())

In [ ]:
for item in test_ingredients:
    quantity, unit = extract_quantity(item)
    print(item, "->", quantity, standardize_unit(unit))

1/2 cup chopped onion -> 0.5 cup
2 tablespoons butter -> 2.0 tbsp
1 1/2 pounds chicken -> 1.5 lb
3 cloves garlic -> 3.0 clove


In [ ]:
def compare_quantities(user_quantity, recipe_quantity,
                       user_unit, recipe_unit):

    user_unit = standardize_unit(user_unit)
    recipe_unit = standardize_unit(recipe_unit)

    if user_unit != recipe_unit:
        return "Unit mismatch"

    if user_quantity >= recipe_quantity:
        return "Sufficient"

    return "Insufficient"

In [ ]:
recipe = next(
    r for r in recipes.values()
    if r.get("title") == "Easy Chicken Cacciatore"
)

for ingredient in recipe["ingredients"]:
    quantity, unit = extract_quantity(ingredient)

    print("Ingredient:", ingredient)
    print("Quantity:", quantity)
    print("Unit:", standardize_unit(unit))
    print()

Ingredient: 1/3 cup chopped onion ADVERTISEMENT
Quantity: 0.3333333333333333
Unit: cup

Ingredient: 1 clove garlic, chopped ADVERTISEMENT
Quantity: 1.0
Unit: clove

Ingredient: 1/3 cup chopped green bell pepper ADVERTISEMENT
Quantity: 0.3333333333333333
Unit: cup

Ingredient: 3/4 pound chicken meat, cooked and cubed ADVERTISEMENT
Quantity: 0.75
Unit: lb

Ingredient: 1/2 cup whole peeled tomatoes ADVERTISEMENT
Quantity: 0.5
Unit: cup

Ingredient: 1/2 cup green beans ADVERTISEMENT
Quantity: 0.5
Unit: cup

Ingredient: 1/4 teaspoon dried oregano ADVERTISEMENT
Quantity: 0.25
Unit: tsp

Ingredient: ADVERTISEMENT
Quantity: None
Unit: None



In [ ]:
def parse_ingredient(ingredient):
    if not isinstance(ingredient, str):
        return None

    ingredient = re.sub(
        r'advertisement', '', ingredient, flags=re.IGNORECASE
    ).strip()

    if not ingredient:
        return None

    quantity, unit = extract_quantity(ingredient)

    remaining = re.sub(
        r'^\s*(\d+\s+\d+/\d+|\d+/\d+|\d+(?:\.\d+)?)\s*',
        '',
        ingredient
    ) if quantity is not None else ingredient

    if unit:
        remaining = re.sub(
            rf'^\s*{re.escape(unit)}\b',
            '',
            remaining,
            flags=re.IGNORECASE
        )

    name = normalize_ingredient(remaining)

    if not name:
        return None

    return {
        "name": name,
        "quantity": quantity,
        "unit": standardize_unit(unit)
    }

In [ ]:
test_ingredients = [
    "2 cups chopped onion",
    "3 cloves garlic",
    "salt to taste",
    "fresh coriander",
    "ADVERTISEMENT"
]

for ingredient in test_ingredients:
    print(ingredient, "->", parse_ingredient(ingredient))

2 cups chopped onion -> {'name': 'onion', 'quantity': 2.0, 'unit': 'cup'}
3 cloves garlic -> {'name': 'garlic', 'quantity': 3.0, 'unit': 'clove'}
salt to taste -> {'name': 'salt to taste', 'quantity': None, 'unit': None}
fresh coriander -> {'name': 'fresh coriander', 'quantity': None, 'unit': None}
ADVERTISEMENT -> None


In [ ]:
def check_recipe_ingredients(user_ingredients, recipe_ingredients):
    user_data = [
        parse_ingredient(item) for item in user_ingredients
    ]

    recipe_data = [
        parse_ingredient(item) for item in recipe_ingredients
    ]

    user_data = [item for item in user_data if item]
    recipe_data = [item for item in recipe_data if item]

    results = []

    for required in recipe_data:
        available = next(
            (
                item for item in user_data
                if ingredient_matches(
                    item["name"], required["name"]
                )
            ),
            None
        )

        if available is None:
            status = "Not available"

        elif (
            available["quantity"] is None
            or required["quantity"] is None
        ):
            status = "Quantity unknown"

        else:
            status = compare_quantities(
                available["quantity"],
                required["quantity"],
                available["unit"],
                required["unit"]
            )

        results.append({
            "ingredient": required["name"],
            "required": required["quantity"],
            "unit": required["unit"],
            "status": status
        })

    return results

In [ ]:
recipe = next(
    r for r in recipes.values()
    if r.get("title") == "Easy Chicken Cacciatore"
)

comparison = check_recipe_ingredients(
    user_ingredients,
    recipe["ingredients"]
)

for item in comparison:
    print(item)

{'ingredient': 'onion', 'required': 0.3333333333333333, 'unit': 'cup', 'status': 'Sufficient'}
{'ingredient': 'garlic', 'required': 1.0, 'unit': 'clove', 'status': 'Sufficient'}
{'ingredient': 'green bell pepper', 'required': 0.3333333333333333, 'unit': 'cup', 'status': 'Not available'}
{'ingredient': 'chicken meat cooked and cubed', 'required': 0.75, 'unit': 'lb', 'status': 'Sufficient'}
{'ingredient': 'whole peeled tomatoes', 'required': 0.5, 'unit': 'cup', 'status': 'Sufficient'}
{'ingredient': 'green beans', 'required': 0.5, 'unit': 'cup', 'status': 'Not available'}
{'ingredient': 'dried oregano', 'required': 0.25, 'unit': 'tsp', 'status': 'Not available'}


In [ ]:
def summarize_ingredient_status(comparison):
    sufficient = sum(
        item["status"] == "Sufficient" for item in comparison
    )

    insufficient = sum(
        item["status"] == "Insufficient" for item in comparison
    )

    unavailable = sum(
        item["status"] == "Not available" for item in comparison
    )

    unknown = sum(
        item["status"] == "Quantity unknown" for item in comparison
    )

    return {
        "sufficient": sufficient,
        "insufficient": insufficient,
        "unavailable": unavailable,
        "unknown": unknown,
        "can_prepare": (
            insufficient == 0
            and unavailable == 0
            and unknown == 0
        )
    }

In [ ]:
summary = summarize_ingredient_status(comparison)

print(summary)

{'sufficient': 1, 'insufficient': 3, 'unavailable': 3, 'can_prepare': False}


In [ ]:
def recommend_recipes_with_quantities(user_ingredients, limit=5):
    results = match_recipes_by_ingredients(
        user_ingredients,
        limit=limit
    )

    recommendations = []

    for result in results:
        recipe = recipes[result["id"]]

        comparison = check_recipe_ingredients(
            user_ingredients,
            recipe.get("ingredients") or []
        )

        summary = summarize_ingredient_status(comparison)

        sufficient = [
            item["ingredient"]
            for item in comparison
            if item["status"] == "Sufficient"
        ]

        insufficient = [
            item["ingredient"]
            for item in comparison
            if item["status"] == "Insufficient"
        ]

        unavailable = [
            item["ingredient"]
            for item in comparison
            if item["status"] == "Not available"
        ]

        unknown = [
            item["ingredient"]
            for item in comparison
            if item["status"] == "Quantity unknown"
        ]

        recommendations.append({
            "id": result["id"],
            "title": result["title"],
            "match_score": result["match_score"],
            "matched_ingredients": result["matched_ingredients"],
            "sufficient_ingredients": sufficient,
            "insufficient_ingredients": insufficient,
            "unavailable_ingredients": unavailable,
            "unknown_quantity_ingredients": unknown,
            "ingredient_status": comparison,
            "summary": summary
        })

    recommendations.sort(
        key=lambda x: (
            x["summary"]["sufficient"],
            -x["summary"]["insufficient"],
            -x["summary"]["unavailable"],
            x["match_score"]
        ),
        reverse=True
    )

    return recommendations

In [ ]:
def ingredient_matches(user_item, recipe_item):
    user_words = set(user_item.split())
    recipe_words = set(recipe_item.split())

    if not user_words or not recipe_words:
        return False

    exclusions = {
        "chicken": ["broth", "stock", "bouillon"],
        "garlic": ["salt", "powder"],
        "onion": ["powder"],
        "tomato": ["sauce", "soup", "paste"]
    }

    for ingredient, excluded in exclusions.items():
        if ingredient in user_words:
            if any(word in recipe_words for word in excluded):
                return False

    variations = {
        "tomato": ["tomato", "tomatoes"],
        "potato": ["potato", "potatoes"]
    }

    for word in user_words:
        allowed = variations.get(word, [word])
        if not any(item in recipe_words for item in allowed):
            return False

    return True

In [ ]:
def is_vegetarian(recipe):
    non_vegetarian_ingredients = [
        "chicken", "beef", "pork", "mutton", "lamb",
        "fish", "salmon", "tuna", "shrimp", "prawn",
        "crab", "lobster", "turkey", "bacon", "ham",
        "sausage", "anchovy", "meat", "duck"
    ]

    title = (recipe.get("title") or "").lower()

    title_keywords = [
        "pot roast", "fisherman", "fish pie"
    ]

    if any(word in title for word in title_keywords):
        return False

    ingredients = recipe.get("normalized_ingredients", [])
    text = " ".join(ingredients).lower()
    words = set(text.split())

    for item in non_vegetarian_ingredients:
        if item in words:
            return False

    return True

In [ ]:
def filter_vegetarian_recipes():
    vegetarian_recipes = []

    for recipe in recipes.values():
        if is_vegetarian(recipe):
            vegetarian_recipes.append(recipe)

    return vegetarian_recipes

In [ ]:
vegetarian_recipes = filter_vegetarian_recipes()

print("Total vegetarian recipes:", len(vegetarian_recipes))
print("Sample recipes:")

for recipe in vegetarian_recipes[:5]:
    print(recipe["title"])

Total vegetarian recipes: 81787
Sample recipes:
Best Chocolate Chip Cookies
Homemade Mac and Cheese Casserole
Banana Banana Bread
Mom's Zucchini Bread
The Best Rolled Sugar Cookies


In [ ]:
test_titles = [
    "Awesome Slow Cooker Pot Roast",
    "Chef John's Fisherman's Pie",
    "Best Chocolate Chip Cookies"
]

for title in test_titles:
    for recipe in recipes.values():
        if recipe["title"] == title:
            print(title, "->", is_vegetarian(recipe))
            break

Awesome Slow Cooker Pot Roast -> False
Chef John's Fisherman's Pie -> False
Best Chocolate Chip Cookies -> True


In [64]:
def filter_restricted_recipes(candidate_recipes, restrictions):
    restriction_keywords = {
        "peanuts": [
            "peanut", "peanuts", "peanut oil",
            "peanut butter", "groundnut"
        ],
        "dairy": [
            "milk", "buttermilk", "cheese", "butter",
            "cream", "yogurt", "curd", "ghee",
            "whey", "casein", "milk powder",
            "condensed milk", "evaporated milk"
        ],
        "eggs": ["egg", "eggs", "egg yolk", "egg white"],
        "chicken": ["chicken"],
        "meat": [
            "beef", "pork", "mutton", "lamb",
            "chicken", "turkey"
        ]
    }

    filtered = []

    for recipe in candidate_recipes:
        ingredients = recipe.get("normalized_ingredients") or []

        text = " ".join(
            str(item).lower() for item in ingredients if item
        )

        excluded = False

        for restriction in restrictions:
            keywords = restriction_keywords.get(
                restriction.lower(), [restriction.lower()]
            )

            for keyword in keywords:
                pattern = r'\b' + re.escape(keyword) + r'\b'

                if re.search(pattern, text):
                    excluded = True
                    break

            if excluded:
                break

        if not excluded:
            filtered.append(recipe)

    return filtered


recipemate_clean.filter_restricted_recipes = filter_restricted_recipes

In [66]:
print(recipemate_clean.chatbot_response(
    "What can I make with moonflower petals and unicorn root without dairy?"
))

I couldn't find recipes matching your ingredients and restrictions.


In [67]:
print(recipemate_clean.chatbot_response(
    "What can I use instead of butter?"
))

Substitution suggestions for butter:

Recipe: Homemade Croutons 
Substitute: fruity extra-virgin olive oil
Suggested quantity: 1 1/2 tablespoons
Evidence: I often substitute fruity extra-virgin olive oil for butter, sometimes plain, sometimes mixed with 1/4 teaspoon crushed garlic.
Recipe ID: recipes_raw_nosource_epi_Dc.UQ0JKGirLkWeTp8VlsWGokBX2xNy




In [ ]:
for recipe in filtered_multiple:
    ingredients = " ".join(
        recipe.get("normalized_ingredients", [])
    ).lower()

    if any(word in ingredients for word in [
        "peanut", "milk", "cheese", "butter",
        "cream", "yogurt", "curd", "ghee"
    ]):
        print("Possible restricted ingredient:", recipe["title"])

In [ ]:
def match_recipes_by_ingredients(
    user_ingredients,
    limit=5,
    restrictions=None
):
    results = []

    for recipe in recipes.values():

        filtered = filter_restricted_recipes(
            [recipe],
            restrictions or []
        )

        if not filtered:
            continue

        recipe_ingredients = recipe.get(
            "normalized_ingredients", []
        )

        matched = []
        missing = []

        for ingredient in user_ingredients:
            if any(
                ingredient_matches(
                    normalize_ingredient(ingredient),
                    item
                )
                for item in recipe_ingredients
            ):
                matched.append(ingredient)
            else:
                missing.append(ingredient)

        if not matched:
            continue

        match_score = len(matched) / len(user_ingredients)

        results.append({
            "id": recipe["id"],
            "title": recipe["title"],
            "matched_ingredients": matched,
            "missing_ingredients": missing,
            "match_score": match_score
        })

    results.sort(
        key=lambda x: x["match_score"],
        reverse=True
    )

    return results[:limit]

In [ ]:
def find_substitutions(target_ingredient):
    results = []

    for recipe in recipes.values():
        ingredients = recipe.get("ingredients") or []
        instructions = recipe.get("instructions") or ""

        if isinstance(instructions, list):
            instructions = " ".join(instructions)

        evidence_text = []

        for item in ingredients:
            evidence_text.append(str(item))

        evidence_text.append(str(instructions))

        for line in evidence_text:
            line_lower = line.lower()

            if (
                target_ingredient.lower() in line_lower
                and (
                    "substitute" in line_lower
                    or "instead of" in line_lower
                    or "or use" in line_lower
                )
                and "no substitutes" not in line_lower
            ):
                results.append({
                    "id": recipe["id"],
                    "title": recipe["title"],
                    "evidence": line.strip()
                })

    unique_results = {
        (item["id"], item["evidence"]): item
        for item in results
    }

    return list(unique_results.values())

In [ ]:
butter_results = find_substitutions("butter")

for item in butter_results:
    if item["id"] == "recipes_raw_nosource_epi_Dc.UQ0JKGirLkWeTp8VlsWGokBX2xNy":
        print(item["title"])
        print(item["evidence"])

Homemade Croutons 
Preheat oven to 300°F. Trim crusts from bread and discard. Lightly butter both sides of each slice (or, if you prefer, first blend butter with crushed garlic). I often substitute fruity extra-virgin olive oil for butter, sometimes plain, sometimes mixed with 1/4 teaspoon crushed garlic. Because bread absorbs more oil than softened butter, 1 1/2 tablespoons oil is about right. Cut each buttered slice into 1/3-inch cubes and spread in a single layer, not touching, on an ungreased large baking sheet. Slide onto the middle oven shelf and toast about 15 minutes until pale tan. Use croutons as the recipe directs.
Preheat oven to 300°F.
Trim crusts from bread and discard.
Lightly butter both sides of each slice (or, if you prefer, first blend butter with crushed garlic). I often substitute fruity extra-virgin olive oil for butter, sometimes plain, sometimes mixed with 1/4 teaspoon crushed garlic. Because bread absorbs more oil than softened butter, 1 1/2 tablespoons oil is 

In [ ]:
oil_results = find_substitutions("olive oil")

print("Olive oil candidates:", len(oil_results))

for item in oil_results[:5]:
    print("\nRecipe:", item["title"])
    print("Recipe ID:", item["id"])
    print("Evidence:", item["evidence"])

Olive oil candidates: 251

Recipe: Pumpkin Cream Cheese Muffins
Recipe ID: recipes_raw_nosource_ar_Bu6Ay9BW8UxXUeqzxiI0YHnVvMGsXce
Evidence: Preheat oven to 375 degrees F (190 degrees C). Grease and flour 18 muffin cups, or use paper liners.
To make the filling: In a medium bowl, beat cream cheese until soft. Add egg, vanilla and brown sugar. Beat until smooth, then set aside.
For the streusel topping: In a medium bowl, mix flour, sugar, cinnamon and pecans. Add butter and cut it in with a fork until crumbly. Set aside.
For the muffin batter: In a large bowl, sift together flour, sugar, baking powder, cinnamon and salt. Make a well in the center of flour mixture and add eggs, pumpkin, olive oil and vanilla. Beat together until smooth.
Place pumpkin mixture in muffin cups about 1/2 full. Then add one tablespoon of the cream cheese mixture right in the middle of the batter. Try to keep cream cheese from touching the paper cup. Sprinkle on the streusel topping.
Bake at 375 degrees F (195 

In [1]:
import shutil
import os

source = "/content/copy_of_recipe_chatbot_genai_assignment.py"
destination = "/content/drive/MyDrive/Recipe_Chatbot/recipemate.py"

if os.path.exists(source):
    shutil.copy2(source, destination)
    print("RecipeMate functions backed up successfully!")
    print("Saved at:", destination)
else:
    print("File not found. Check the filename in Colab's Files panel.")

File not found. Check the filename in Colab's Files panel.


In [2]:
import os

print("Python files in /content:")
for file in os.listdir("/content"):
    if file.endswith(".py"):
        print(file)

Python files in /content:


In [3]:
from google.colab import files

uploaded = files.upload()

Saving copy_of_recipe_chatbot_genai_assignment.py to copy_of_recipe_chatbot_genai_assignment.py


In [5]:
import shutil
import os

source = "/content/" + next(iter(uploaded))
destination = "/content/drive/MyDrive/Recipe_Chatbot/recipemate.py"

# Ensure the destination directory exists before copying
os.makedirs(os.path.dirname(destination), exist_ok=True)

shutil.copy2(source, destination)

print("RecipeMate functions backed up successfully!")
print("Saved at:", destination)
print("File size:", os.path.getsize(destination), "bytes")

RecipeMate functions backed up successfully!
Saved at: /content/drive/MyDrive/Recipe_Chatbot/recipemate.py
File size: 55288 bytes


In [6]:
import os

path = "/content/drive/MyDrive/Recipe_Chatbot/recipemate.py"

print("File exists:", os.path.exists(path))
print("File size:", os.path.getsize(path), "bytes")

File exists: True
File size: 55288 bytes


In [9]:
%pip install -q -U google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.0.0 which is incompatible.


In [10]:
%pip install -q "google-genai>=2.12.1,<3" "google-auth==2.49.0"

In [11]:
import google.genai
import google.auth

print("Gemini SDK version:", google.genai.__version__)
print("Google Auth version:", google.auth.__version__)

Gemini SDK version: 2.12.1
Google Auth version: 2.49.0


In [12]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [14]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Suggest a simple recipe using tomatoes and onions."
)

print(response.text)

Here is a recipe for **Rustic Stewed Tomatoes and Onions**. It requires minimal ingredients, takes about 20 minutes, and turns basic pantry items into a rich, savory dish.

---

### **Rustic Stewed Tomatoes and Onions**

* **Prep time:** 5 minutes
* **Cook time:** 15–20 minutes
* **Servings:** 2–3

#### **Ingredients:**
* **1 large onion** (yellow or red), thinly sliced
* **4–5 medium ripe tomatoes**, roughly chopped (or 1 can of whole/diced tomatoes)
* **2 tbsp olive oil** (or butter)
* **1 clove garlic**, minced *(optional)*
* **Salt and black pepper**, to taste
* **Optional finishing touches:** A pinch of dried oregano, red pepper flakes, fresh basil, or a tiny pinch of sugar (if tomatoes are too acidic).

---

#### **Instructions:**

1. **Cook the onions:**  
   Heat the olive oil in a skillet over medium heat. Add the sliced onions and a pinch of salt. Cook for about 6–8 minutes, stirring occasionally, until the onions are soft, translucent, and just beginning to turn golden aroun

In [88]:
def answer_recipe_question(message):
    recipe_id = chat_history["selected_recipe_id"]

    if not recipe_id:
        return "Please select a recipe first."

    recipe = recipes.get(recipe_id)

    if not recipe:
        return "Recipe details are unavailable."

    instructions = recipe.get("instructions") or ""

    if isinstance(instructions, list):
        instructions = " ".join(instructions)

    sentences = re.split(r'(?<=[.!?])\s+', str(instructions))

    time_pattern = (
        r'\b\d+(?:\s+\d+/\d+)?'
        r'(?:\s+to\s+\d+(?:\s+\d+/\d+)?)?'
        r'\s*(?:seconds?|secs?|minutes?|mins?|hours?|hrs?)\b'
    )

    cooking_pattern = (
        r'\b(cook|bake|roast|simmer|heat|soften|fry|boil|'
        r'microwave|grill|broil|steam)\b'
    )

    relevant = []

    for sentence in sentences:
        if re.search(time_pattern, sentence, re.I):
            if re.search(cooking_pattern, sentence, re.I):
                if sentence.strip() not in relevant:
                    relevant.append(sentence.strip())

    if relevant:
        return "Cooking times from the recipe:\n\n" + "\n".join(
            "- " + sentence for sentence in relevant
        )

    return "I couldn't find a specific cooking time in this recipe."

In [89]:
recipemate_clean.answer_recipe_question = answer_recipe_question

In [70]:
def substitution_response(target_ingredient):
    results = get_substitution_recommendations(target_ingredient)

    if not results:
        return (
            f"I couldn't find verified substitution evidence "
            f"for {target_ingredient} in the recipe dataset."
        )

    response = f"Substitution suggestions for {target_ingredient}:\n\n"

    for item in results[:5]:
        response += f"Recipe: {item['recipe_title']}\n"
        response += f"Substitute: {item['substitute_ingredient']}\n"

        quantity = item["suggested_quantity"]

        if quantity:
            response += (
                f"Suggested quantity for this recipe: "
                f"{quantity['quantity']} {quantity['unit']}\n"
            )
            response += (
                "Note: This quantity is specific to this recipe "
                "and may not apply to other recipes.\n"
            )
        else:
            response += "Suggested quantity: Not specified\n"

        response += f"Evidence: {item['evidence']}\n"
        response += f"Recipe ID: {item['recipe_id']}\n\n"

    return response

In [71]:
recipemate_clean.substitution_response = substitution_response

In [72]:
print(recipemate_clean.chatbot_response(
    "What can I use instead of butter?"
))

Substitution suggestions for butter:

Recipe: Homemade Croutons 
Substitute: fruity extra-virgin olive oil
Suggested quantity for this recipe: 1 1/2 tablespoons
Note: This quantity is specific to this recipe and may not apply to other recipes.
Evidence: I often substitute fruity extra-virgin olive oil for butter, sometimes plain, sometimes mixed with 1/4 teaspoon crushed garlic.
Recipe ID: recipes_raw_nosource_epi_Dc.UQ0JKGirLkWeTp8VlsWGokBX2xNy




In [73]:
print(recipemate_clean.chatbot_response(
    "What are the 10 most commonly used ingredients?"
))

Top 10 most common ingredients:
- salt: 29607
- garlic: 19202
- olive oil: 18897
- sugar: 18383
- all-purpose flour: 17291
- water: 14168
- kosher salt: 12456
- butter: 12344
- white sugar: 10893
- vanilla extract: 10538



In [74]:
print(recipemate_clean.chatbot_response(
    "What is the average and median number of ingredients per recipe?"
))

Average ingredients per recipe: 10.52
Median ingredients per recipe: 10.0


In [16]:
import importlib.util

path = "/content/drive/MyDrive/Recipe_Chatbot/recipemate.py"

spec = importlib.util.spec_from_file_location("recipemate", path)
recipemate = importlib.util.module_from_spec(spec)

In [17]:
import ast

path = "/content/drive/MyDrive/Recipe_Chatbot/recipemate.py"

with open(path, "r", encoding="utf-8") as file:
    code = file.read()

# Ignore Colab-specific commands for this inspection
clean_lines = [
    line for line in code.splitlines()
    if not line.strip().startswith(("!", "%"))
]

tree = ast.parse("\n".join(clean_lines))

for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
        print(node.name, "— line", node.lineno)

semantic_search — line 91
normalize_ingredient — line 142
search_recipes — line 207
hybrid_search — line 325
match_recipes_by_ingredients — line 373
extract_quantity — line 462
standardize_unit — line 515
compare_quantities — line 521
ingredient_matches — line 544
parse_ingredient — line 576
check_recipe_ingredients — line 615
summarize_ingredient_status — line 666
recommend_recipes_with_quantities — line 697
is_vegetarian — line 769
filter_vegetarian_recipes — line 799
filter_restricted_recipes — line 808
find_substitutions — line 875
extract_substitute — line 903
extract_substitution_quantity — line 922
get_substitution_recommendations — line 939
chatbot_response — line 973
recipe_search_response — line 1078
handle_follow_up — line 1095
check_previous_recipe_ingredients — line 1127
get_recipe_details — line 1165
ingredient_match_response — line 1182
substitution_response — line 1203
answer_recipe_question — line 1242
get_total_recipe_count — line 1281
get_common_ingredients — line 12

In [18]:
import ast
from pathlib import Path

source_path = Path("/content/drive/MyDrive/Recipe_Chatbot/recipemate.py")
backup_path = Path("/content/drive/MyDrive/Recipe_Chatbot/recipemate_original.py")

# Preserve the original exported notebook
if not backup_path.exists():
    import shutil
    shutil.copy2(source_path, backup_path)

code = source_path.read_text(encoding="utf-8")
lines = code.splitlines(keepends=True)

tree = ast.parse("\n".join(
    line for line in code.splitlines()
    if not line.strip().startswith(("!", "%"))
))

selected = []

for node in tree.body:
    if isinstance(node, ast.FunctionDef):
        if node.name not in ["home", "chat", "run_flask", "updated_home"]:
            selected.append("".join(lines[node.lineno - 1:node.end_lineno]))

print("Functions identified:", len(selected))
print("Original backup preserved:", backup_path.exists())

Functions identified: 34
Original backup preserved: True


In [19]:
import ast

tree = ast.parse("\n".join(
    line for line in code.splitlines()
    if not line.strip().startswith(("!", "%"))
))

for node in tree.body:
    if isinstance(node, ast.Import):
        print(ast.unparse(node))

    elif isinstance(node, ast.ImportFrom):
        print(ast.unparse(node))

from google.colab import files
import zipfile
import os
import json
import os
from google.colab import drive
import os
import pickle
import faiss
from sentence_transformers import SentenceTransformer
import re
import pickle
import os
import re
from collections import Counter
import statistics
import re
from flask import Flask, request, jsonify
from pathlib import Path
from flask import render_template
from google.colab.output import eval_js
from threading import Thread
from pathlib import Path
import os
from pathlib import Path
from flask import render_template
from pathlib import Path
import os
import os
import pickle
import faiss
from google.colab import drive
from sentence_transformers import SentenceTransformer
import pickle
import os
from pathlib import Path


In [20]:
import ast

tree = ast.parse("\n".join(
    line for line in code.splitlines()
    if not line.strip().startswith(("!", "%"))
))

for node in tree.body:
    if isinstance(node, (ast.Assign, ast.AnnAssign)):
        print(ast.unparse(node))

uploaded = files.upload()
zip_path = 'recipes_raw.zip'
data_path = '/content/data'
recipes = {}
folder_path = '/content/drive/MyDrive/Recipe_Chatbot'
folder_path = '/content/drive/MyDrive/Recipe_Chatbot'
index = faiss.read_index(os.path.join(folder_path, 'recipe_faiss.index'))
model = SentenceTransformer('all-MiniLM-L6-v2')
results = semantic_search('chicken recipes with tomato', 5)
results = semantic_search('chicken recipes with tomato')
units = ['tablespoons', 'tablespoon', 'tbsp', 'teaspoons', 'teaspoon', 'tsp', 'cups', 'cup', 'ounces', 'ounce', 'oz', 'pounds', 'pound', 'lbs', 'lb', 'grams', 'gram', 'g', 'kilograms', 'kilogram', 'kg', 'milliliters', 'milliliter', 'ml', 'liters', 'liter', 'l', 'cloves', 'clove']
containers = ['cans', 'can', 'packages', 'package', 'jars', 'jar', 'bottles', 'bottle', 'boxes', 'box']
save_path = '/content/drive/MyDrive/Recipe_Chatbot/processed_recipes.pkl'
results = search_recipes('chicken tomato')
recipe_ids = []
recipe_texts = []
embeddings = model.en

In [21]:
import ast
from pathlib import Path

clean_lines = [
    line for line in code.splitlines(keepends=True)
    if not line.strip().startswith(("!", "%"))
]

clean_code = "".join(clean_lines)
tree = ast.parse(clean_code)

function_names = [
    node.name for node in tree.body
    if isinstance(node, ast.FunctionDef)
    and node.name not in ["home", "chat", "run_flask", "updated_home"]
]

function_code = []

for node in tree.body:
    if isinstance(node, ast.FunctionDef) and node.name in function_names:
        function_code.append(
            "".join(clean_lines[node.lineno - 1:node.end_lineno])
        )

constants = []

for node in tree.body:
    if isinstance(node, ast.Assign):
        names = [
            target.id for target in node.targets
            if isinstance(target, ast.Name)
        ]

        if any(name in ["units", "containers", "unit_conversions"] for name in names):
            constants.append(
                "".join(clean_lines[node.lineno - 1:node.end_lineno])
            )

module_code = """
import os
import re
import pickle
import statistics
import faiss

from collections import Counter
from sentence_transformers import SentenceTransformer

chat_history = {
    "last_search_results": [],
    "selected_recipe_id": None,
    "last_user_ingredients": [],
    "last_restrictions": []
}

""" + "\n".join(constants) + "\n\n" + "\n\n".join(function_code)

output_path = Path(
    "/content/drive/MyDrive/Recipe_Chatbot/recipemate_clean.py"
)

output_path.write_text(module_code, encoding="utf-8")

print("Clean module created!")
print("Functions included:", len(function_names))
print("Saved:", output_path)

Clean module created!
Functions included: 34
Saved: /content/drive/MyDrive/Recipe_Chatbot/recipemate_clean.py


In [22]:
import ast

path = "/content/drive/MyDrive/Recipe_Chatbot/recipemate_clean.py"

with open(path, "r", encoding="utf-8") as file:
    module_code = file.read()

try:
    ast.parse(module_code)
    print("Syntax validation successful!")
    print("Total lines:", len(module_code.splitlines()))
except SyntaxError as error:
    print("Syntax error:", error)

Syntax validation successful!
Total lines: 1238


In [24]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.1 MB/s  0:00:00


In [25]:
import sys

folder = "/content/drive/MyDrive/Recipe_Chatbot"
if folder not in sys.path:
    sys.path.append(folder)

import recipemate_clean

print("RecipeMate module imported successfully!")
print("Chatbot function available:", hasattr(recipemate_clean, "chatbot_response"))

RecipeMate module imported successfully!
Chatbot function available: True


In [26]:
recipemate_clean.recipes = recipes
recipemate_clean.model = model
recipemate_clean.index = index
recipemate_clean.recipe_ids = recipe_ids
recipemate_clean.substitution_index = substitution_index

print("RecipeMate data connected successfully!")
print("Recipes:", len(recipemate_clean.recipes))
print("FAISS index:", recipemate_clean.index.ntotal)

NameError: name 'recipes' is not defined

In [27]:
import os
import pickle
import faiss
from sentence_transformers import SentenceTransformer

folder = "/content/drive/MyDrive/Recipe_Chatbot"

with open(os.path.join(folder, "processed_recipes.pkl"), "rb") as f:
    recipes = pickle.load(f)

with open(os.path.join(folder, "recipe_embeddings.pkl"), "rb") as f:
    embeddings = pickle.load(f)

index = faiss.read_index(os.path.join(folder, "recipe_faiss.index"))

with open(os.path.join(folder, "recipe_ids.pkl"), "rb") as f:
    recipe_ids = pickle.load(f)

with open(os.path.join(folder, "substitution_index.pkl"), "rb") as f:
    substitution_index = pickle.load(f)

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Data restored successfully!")
print("Recipes:", len(recipes))
print("FAISS index:", index.ntotal)
print("Substitution records:", len(substitution_index))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Recipe_Chatbot/processed_recipes.pkl'

In [28]:
import os
from google.colab import drive

drive.mount("/content/drive")

folder = "/content/drive/MyDrive/Recipe_Chatbot"

print("Folder exists:", os.path.exists(folder))

if os.path.exists(folder):
    print("Files available:")
    for file in os.listdir(folder):
        print(file)

ValueError: Mountpoint must not already contain files

In [29]:
import os

print("Drive path exists:", os.path.exists("/content/drive"))
print("MyDrive exists:", os.path.exists("/content/drive/MyDrive"))
print("RecipeMate folder exists:", os.path.exists("/content/drive/MyDrive/Recipe_Chatbot"))

Drive path exists: True
MyDrive exists: True
RecipeMate folder exists: True


In [30]:
import os

folder = "/content/drive/MyDrive/Recipe_Chatbot"

for file in sorted(os.listdir(folder)):
    path = os.path.join(folder, file)
    if os.path.isfile(path):
        print(f"{file}  |  {os.path.getsize(path) / (1024 * 1024):.2f} MB")
    else:
        print(f"{file}/")

__pycache__/
recipemate.py  |  0.05 MB
recipemate_clean.py  |  0.03 MB
recipemate_original.py  |  0.05 MB


In [31]:
import os

search_names = {
    "processed_recipes.pkl",
    "recipe_embeddings.pkl",
    "recipe_faiss.index",
    "recipe_ids.pkl",
    "substitution_index.pkl"
}

print("Searching Google Drive...")

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file in search_names:
            print(os.path.join(root, file))

Searching Google Drive...


In [32]:
import os

paths = [
    "/content/data",
    "/content/recipes_raw.zip",
    "/content/drive/MyDrive/Recipe_Chatbot"
]

for path in paths:
    print(f"\n{path}")
    print("Exists:", os.path.exists(path))

    if os.path.isdir(path):
        print("Contents:", os.listdir(path)[:10])


/content/data
Exists: False

/content/recipes_raw.zip
Exists: False

/content/drive/MyDrive/Recipe_Chatbot
Exists: True
Contents: ['recipemate_original.py', 'recipemate.py', 'recipemate_clean.py', '__pycache__']


In [75]:
evaluation_tests = [
    {
        "test": "Recipe search",
        "input": "Suggest a chocolate cake recipe",
        "expected": "Returns relevant chocolate cake recipes"
    },
    {
        "test": "Follow-up context",
        "input": "How long should I bake the first recipe?",
        "expected": "Answers using the previously selected recipe"
    },
    {
        "test": "Ingredient matching with restriction",
        "input": "What can I make with chicken and tomatoes without dairy?",
        "expected": "Returns recipes matching the ingredients and excluding dairy keywords"
    },
    {
        "test": "No-match query",
        "input": "What can I make with moonflower petals and unicorn root without dairy?",
        "expected": "Clearly reports that no matching recipe was found"
    },
    {
        "test": "Dataset statistics",
        "input": "What are the 10 most commonly used ingredients?",
        "expected": "Returns the top 10 ingredients with their counts"
    },
    {
        "test": "Substitution evidence",
        "input": "What can I use instead of butter?",
        "expected": "Returns a dataset-backed substitution with recipe-specific evidence"
    },
    {
        "test": "Missing substitution evidence",
        "input": "What can I use instead of saffron?",
        "expected": "Clearly reports when verified evidence is unavailable"
    }
]

print("Total evaluation tests:", len(evaluation_tests))

for i, test in enumerate(evaluation_tests, 1):
    print(f"{i}. {test['test']}")

Total evaluation tests: 7
1. Recipe search
2. Follow-up context
3. Ingredient matching with restriction
4. No-match query
5. Dataset statistics
6. Substitution evidence
7. Missing substitution evidence


In [76]:
evaluation_results = []

for test in evaluation_tests:
    response = recipemate_clean.chatbot_response(test["input"])

    evaluation_results.append({
        "test": test["test"],
        "input": test["input"],
        "expected": test["expected"],
        "actual_output": response
    })

print("Evaluation execution completed!")

for result in evaluation_results:
    print("\nTest:", result["test"])
    print("Input:", result["input"])
    print("Actual output:", result["actual_output"])
    print("-" * 60)

Evaluation execution completed!

Test: Recipe search
Input: Suggest a chocolate cake recipe
Actual output: Here are some recipes I found:

1. Chocolate Cake for Two
   Recipe ID: recipes_raw_nosource_fn_csytfeg7551EaFM/gGTgefjtjPayK/y

2. Chocolate Cake in a Mug
   Recipe ID: recipes_raw_nosource_fn_TmEQYD2zYGpD3yczwMpvvtWxHPvKn1q

3. Chocolate Cake in a Mug
   Recipe ID: recipes_raw_nosource_ar_AAp972Buo.yXW.jAOp9bvFNG62IqGgi

4. Chocolate Cake in a Mug
   Recipe ID: recipes_raw_nosource_fn_xJa1Hlp54RZuFLs7U9f79F05Vwwd/Uu

5. Dark Chocolate Cake
   Recipe ID: recipes_raw_nosource_fn_LB10TnNn.jCjrKr.fpY6MRLf.VljNIe


------------------------------------------------------------

Test: Follow-up context
Input: How long should I bake the first recipe?
Actual output: Please select a recipe first.
------------------------------------------------------------

Test: Ingredient matching with restriction
Input: What can I make with chicken and tomatoes without dairy?
Actual output: Here are rec

In [81]:
def recipe_search_response(query):
    results = hybrid_search(query, limit=5)

    chat_history["last_search_results"] = results
    chat_history["selected_recipe_id"] = (
        results[0]["id"] if results else None
    )

    if not results:
        return "I couldn't find any matching recipes in the dataset."

    response = "Here are some recipes I found:\n\n"

    for i, item in enumerate(results, 1):
        response += f"{i}. {item['title']}\n"
        response += f"   Recipe ID: {item['id']}\n\n"

    return response

In [82]:
recipemate_clean.recipe_search_response = recipe_search_response

In [93]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="What is the best substitute for saffron in any recipe? Give a specific replacement and quantity."
)

print(response.text)

The single best substitute for saffron is **ground turmeric**. 

While no spice perfectly replicates saffron’s distinct floral and earthy flavor, turmeric provides the exact same vibrant yellow-gold color and adds a mild, warm earthiness.

### Specific Replacement & Quantity:
* For every **generous pinch of saffron threads** (approx. 10–15 threads or 1/4 teaspoon crushed saffron), use:
  * **1/4 teaspoon of ground turmeric**

---

### *Chef’s Tip for the Best Balance (Color + Flavor):*
Pure turmeric can make a dish neon-yellow and slightly peppery. To mimic saffron's signature red-orange hue and subtle sweetness, the ultimate culinary swap is a **1:1 blend of turmeric and sweet paprika**:
* **1/8 teaspoon ground turmeric + 1/8 teaspoon sweet paprika** (to replace 1 pinch / 1/4 tsp saffron threads).


In [94]:
import os

folder = "/content/drive/MyDrive/Recipe_Chatbot"

for file in sorted(os.listdir(folder)):
    print(file)

__pycache__
processed_recipes.pkl
recipe_embeddings.pkl
recipe_faiss.index
recipe_ids.pkl
recipemate.py
recipemate_clean.py
recipemate_original.py
substitution_index.pkl


In [95]:
import os

print(os.path.exists("/content/templates/index.html"))

False


In [96]:
import os

for root, dirs, files in os.walk("/content"):
    if "index.html" in files:
        print(os.path.join(root, "index.html"))

In [97]:
import os

os.makedirs("/content/templates", exist_ok=True)

print("RecipeMate templates folder created!")

RecipeMate templates folder created!


In [99]:
import shutil
import os

folder = "/content/drive/MyDrive/Recipe_Chatbot"
os.makedirs(folder, exist_ok=True)

shutil.copy(
    "/content/templates/index.html",
    folder + "/index.html"
)

print("RecipeMate interface backed up to Google Drive!")

RecipeMate interface backed up to Google Drive!


In [101]:
from flask import Flask, request, jsonify, render_template

app = Flask(__name__)
app.secret_key = "recipe_chatbot_secret_key"

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/chat", methods=["POST"])
def chat():
    data = request.get_json()
    message = data.get("message", "").strip()

    if not message:
        return jsonify({"response": "Please enter a message."})

    response = recipemate_clean.chatbot_response(message)
    return jsonify({"response": response})

print("RecipeMate Flask app created!")

RecipeMate Flask app created!


In [102]:
test = app.test_client().get("/")

print("Status:", test.status_code)
print("RecipeMate found:", b"RecipeMate" in test.data)

ERROR:recipemate_clean:Exception on / [GET]
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "/tmp/ipykernel_2071/3077799802.py", line 8, in home
    return render_template("index.html")
  File "/usr/local/lib/python3.13/dist-packages/flask/templating.py", line 150, in render_template
    template = app.jinja_env.get_or_select_te

Status: 500
RecipeMate found: False


In [103]:
import os

print("Colab template exists:", os.path.exists("/content/templates/index.html"))
print("Drive backup exists:", os.path.exists(
    "/content/drive/MyDrive/Recipe_Chatbot/index.html"
))
print("Flask app root:", app.root_path)

Colab template exists: True
Drive backup exists: True
Flask app root: /content/drive/MyDrive/Recipe_Chatbot


In [104]:
app.template_folder = "/content/templates"
app.jinja_loader = None
app.jinja_env.cache.clear()

test = app.test_client().get("/")

print("Status:", test.status_code)
print("RecipeMate found:", b"RecipeMate" in test.data)

ERROR:recipemate_clean:Exception on / [GET]
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
  File "/usr/local/lib/python3.13/dist-packages/flask/app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "/tmp/ipykernel_2071/3077799802.py", line 8, in home
    return render_template("index.html")
  File "/usr/local/lib/python3.13/dist-packages/flask/templating.py", line 150, in render_template
    template = app.jinja_env.get_or_select_te

Status: 500
RecipeMate found: False


In [105]:
from jinja2 import FileSystemLoader
import os

template_path = "/content/templates/index.html"

print("HTML file exists:", os.path.isfile(template_path))

app.jinja_env.loader = FileSystemLoader("/content/templates")
app.jinja_env.cache.clear()

test = app.test_client().get("/")

print("Status:", test.status_code)
print("RecipeMate found:", b"RecipeMate" in test.data)

HTML file exists: True
Status: 200
RecipeMate found: True


In [111]:
%%writefile /content/templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>RecipeMate - Your Smart Cooking Companion</title>

<style>
* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
    font-family: 'Segoe UI', Arial, sans-serif;
}

body {
    display: flex;
    height: 100vh;
    background: #f7f8f5;
    color: #263b30;
}

.sidebar {
    width: 260px;
    min-width: 260px;
    background: #193f30;
    color: white;
    padding: 35px 28px;
    display: flex;
    flex-direction: column;
}

.logo {
    font-size: 28px;
    font-weight: 700;
    margin-bottom: 8px;
}

.tagline {
    font-size: 15px;
    color: #d2e3d7;
}

.sidebar-section {
    margin-top: 65px;
}

.sidebar-section p {
    font-size: 16px;
    line-height: 2.2;
    color: #f0f5f1;
}

.credit {
    margin-top: auto;
    font-size: 14px;
    color: #c6d9cc;
}

.main {
    flex: 1;
    display: flex;
    flex-direction: column;
    min-width: 0;
}

.header {
    padding: 25px 35px;
    background: #ffffff;
    border-bottom: 1px solid #e8ece7;
}

.header h2 {
    font-size: 23px;
    color: #234333;
}

.header p {
    margin-top: 6px;
    color: #829087;
    font-size: 14px;
}

.chat-area {
    flex: 1;
    overflow-y: auto;
    padding: 35px;
    display: flex;
    flex-direction: column;
    gap: 22px;
}

.welcome {
    margin: auto;
    text-align: center;
    max-width: 650px;
}

.welcome-icon {
    font-size: 45px;
    margin-bottom: 15px;
}

.welcome h1 {
    font-size: 30px;
    color: #234333;
    margin-bottom: 12px;
}

.welcome p {
    color: #78877d;
    font-size: 15px;
    line-height: 1.7;
    margin-bottom: 28px;
}

.suggestions {
    display: flex;
    justify-content: center;
    flex-wrap: wrap;
    gap: 12px;
}

.suggestion {
    background: white;
    border: 1px solid #dce6dc;
    border-radius: 14px;
    padding: 15px 18px;
    color: #315a40;
    font-size: 14px;
    cursor: pointer;
    transition: 0.2s;
}

.suggestion:hover {
    background: #eaf2e9;
    border-color: #8cac91;
}

.message {
    max-width: 78%;
    padding: 17px 21px;
    border-radius: 16px;
    font-size: 15px;
    line-height: 1.8;
    white-space: pre-wrap;
    overflow-wrap: anywhere;
}

.user {
    align-self: flex-end;
    background: #dcebdd;
    color: #294434;
    border-bottom-right-radius: 4px;
}

.bot {
    align-self: flex-start;
    background: white;
    border: 1px solid #e3e9e2;
    color: #34443a;
    border-bottom-left-radius: 4px;
}

.input-area {
    display: flex;
    gap: 13px;
    padding: 20px 35px;
    background: white;
    border-top: 1px solid #e8ece7;
}

.input-area input {
    flex: 1;
    min-width: 0;
    padding: 17px 23px;
    border: 1px solid #dce5dc;
    border-radius: 30px;
    outline: none;
    font-size: 15px;
}

.input-area input:focus {
    border-color: #477957;
}

.input-area button {
    padding: 0 30px;
    border: none;
    border-radius: 30px;
    background: #28583b;
    color: white;
    font-size: 15px;
    font-weight: 600;
    cursor: pointer;
}

.input-area button:hover {
    background: #193f30;
}

@media(max-width: 700px) {
    .sidebar {
        display: none;
    }

    .chat-area {
        padding: 20px;
    }

    .input-area {
        padding: 15px;
    }

    .header {
        padding: 20px;
    }

    .message {
        max-width: 95%;
    }
}
</style>
</head>

<body>

<aside class="sidebar">
    <div class="logo">🍃 RecipeMate</div>
    <div class="tagline">Your smart cooking companion</div>

    <div class="sidebar-section">
        <p>
            Discover recipes.<br>
            Match ingredients.<br>
            Find better alternatives.
        </p>
    </div>

    <div class="credit">Built by Karuka Ashritha</div>
</aside>

<main class="main">

    <header class="header">
        <h2>RecipeMate</h2>
        <p>Ask me anything about recipes and cooking.</p>
    </header>

    <section class="chat-area" id="chatArea">

        <div class="welcome" id="welcome">
            <div class="welcome-icon">🥗</div>
            <h1>What would you like to cook?</h1>
            <p>
                Your smart cooking companion is here to help.
                Find recipes based on ingredients you have,
                discover alternatives and explore new dishes.
            </p>

            <div class="suggestions">
                <button class="suggestion"
                    onclick="sendSuggestion('Suggest a chocolate cake recipe')">
                    🍫 Find a recipe
                </button>

                <button class="suggestion"
                    onclick="sendSuggestion('What can I make with chicken and tomatoes?')">
                    🥕 Match my ingredients
                </button>

                <button class="suggestion"
                    onclick="sendSuggestion('What can I use instead of butter?')">
                    🥣 Find substitutes
                </button>

                <button class="suggestion"
                    onclick="sendSuggestion('What are the 10 most commonly used ingredients?')">
                    📊 Recipe statistics
                </button>
            </div>
        </div>

    </section>

    <form class="input-area" id="chatForm">
        <input
            type="text"
            id="userInput"
            placeholder="Ask RecipeMate something..."
            autocomplete="off"
            required
        >
        <button type="submit">Send ➤</button>
    </form>

</main>

<script>
const chatArea = document.getElementById("chatArea");
const input = document.getElementById("userInput");
const form = document.getElementById("chatForm");

function addMessage(message, type) {
    const welcome = document.getElementById("welcome");

    if (welcome) {
        welcome.remove();
    }

    const div = document.createElement("div");
    div.className = "message " + type;
    div.textContent = message;

    chatArea.appendChild(div);
    chatArea.scrollTop = chatArea.scrollHeight;

    return div;
}

async function sendMessage(message) {
    if (!message.trim()) return;

    addMessage(message, "user");
    input.value = "";
    input.disabled = true;

    const loading = addMessage("Thinking...", "bot");

    try {
        const response = await fetch("/chat", {
            method: "POST",
            headers: {
                "Content-Type": "application/json"
            },
            body: JSON.stringify({message: message})
        });

        if (!response.ok) {
            throw new Error("Request failed");
        }

        const data = await response.json();
        loading.textContent = data.response || "No response received.";

    } catch (error) {
        loading.textContent = "Something went wrong. Please try again.";
    }

    input.disabled = false;
    input.focus();
    chatArea.scrollTop = chatArea.scrollHeight;
}

function sendSuggestion(message) {
    sendMessage(message);
}

form.addEventListener("submit", function(event) {
    event.preventDefault();
    sendMessage(input.value.trim());
});
</script>

</body>
</html>


Overwriting /content/templates/index.html


In [113]:
import shutil

shutil.copy(
    "/content/templates/index.html",
    "/content/drive/MyDrive/Recipe_Chatbot/index.html"
)

print("RecipeMate UI backup saved successfully!")

RecipeMate UI backup saved successfully!


In [114]:
app.jinja_env.cache.clear()

test = app.test_client().get("/")

print("Status code:", test.status_code)
print("Updated RecipeMate UI:", b"<h2>RecipeMate</h2>" in test.data)

Status code: 200
Updated RecipeMate UI: True


In [116]:
from google.colab import files

files.download("/content/drive/MyDrive/Recipe_Chatbot/recipemate_clean.py")
files.download("/content/drive/MyDrive/Recipe_Chatbot/index.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>